# B2-022 — Practice p07

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** multivariate-gaussian

## Task

Implement a Cholesky sampler and a spectral factor, then certify them with fixed-seed probes.

**Contract.**
`sample_gaussian(mu, sigma, n, generator)` takes float64 `mu (d,)`, float64 SPD `sigma (d,d)`, an integer `n`, and a `torch.Generator`; it must compute exactly
`L = torch.linalg.cholesky(sigma)`, then `z = torch.randn(n, d, generator=generator, dtype=torch.float64)`, and return `mu + z @ L.T`, float64 `(n,d)`.
If `torch.linalg.cholesky` fails (non-SPD input), raise `ValueError`.
`spectral_factor(sigma)` returns `Q @ torch.diag(evals.sqrt())` from `evals, Q = torch.linalg.eigh(sigma)`.

**Banned (zero points if used):** `torch.distributions`, `torch.normal`, NumPy random generators, and Python loops over samples.

**Probes** (use the literals below; every generator is a fresh `torch.Generator().manual_seed(SEED)`):

1. With `n = 200_000`: every coordinate of the sample mean is within `0.02` (absolute) of `mu`, and every entry of the $1/n$ sample covariance is within `0.03` (absolute) of `sigma`.
2. Two calls with two fresh generators return `torch.equal` samples.
3. The first sample row equals `mu + L @ z0`, where `z0` is the first row of `torch.randn(200_000, 3, generator=<fresh>, dtype=torch.float64)`, with `ATOL = RTOL = 1e-12`; numerically it is about `(2.101807, -2.067620, 0.211849)` (`atol=1e-6`).
4. `S = spectral_factor(sigma)` satisfies `S @ S.T` $\approx$ `sigma` with `ATOL = RTOL = 1e-10`; samples `mu + z @ S.T` from the same `z` pass probe 1's tolerances, yet differ from the Cholesky samples (they are not `allclose` at `atol=1e-3`).
5. `sigma_not_spd` raises `ValueError`.

Explain in two sentences why probe 1 uses tolerances tied to $n$ while probes 2–3 are exact.

**Required answer-check assertions.** Shapes and dtypes; probes 1–5.

In [ ]:
import torch

SEED = 20260927
mu = torch.tensor([2.0, -1.0, 0.0], dtype=torch.float64)
sigma = torch.tensor([[1.0, 0.8, 0.0],
                      [0.8, 2.0, -0.5],
                      [0.0, -0.5, 0.5]], dtype=torch.float64)
sigma_not_spd = torch.tensor([[1.0, 2.0, 0.0], [2.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=torch.float64)
N_SAMPLES = 200_000
MEAN_ATOL = 0.02
COV_ATOL = 0.03

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.